In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score,mean_absolute_error
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

## Load file

In [2]:
X_Uncleaned = pd.read_csv("../X.csv")
y_Uncleaned = pd.read_csv("../y.csv")

X = X_Uncleaned.iloc[:, 1:]  # remove index
y = y_Uncleaned.iloc[:, 1:]  # remove index

# display(y.head())

In [3]:
loadPickleFile = pd.read_pickle("../rf_model.pkl")
# print(loadPickleFile)

### Preprocess X and Y

In [4]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


### Build Model

In [5]:
# Extract the hyperparams
params = (
    loadPickleFile.get_params()
)  # --> Used to get paramters of a fresh model (not trained yet)

### Baseline Training

In [6]:
model_baseline = RandomForestRegressor(**params)
model_baseline.fit(X_train, y_train)

y_pred_scores_baseline = model_baseline.predict(X_test)

/opt/miniconda3/lib/python3.12/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


### Baseline Model validation

In [7]:
# MSE
mse_baseline = mean_squared_error(y_test, y_pred_scores_baseline)
print("MSE:", mse_baseline)

# MAE
mae_baseline = mean_absolute_error(y_test, y_pred_scores_baseline)
print("MAE:", mae_baseline)

# R-squared
r2_baseline = r2_score(y_test, y_pred_scores_baseline)
print("R² Score:", r2_baseline)

MSE: 3.613336293079118
MAE: 1.5479255111866503
R² Score: 0.8890811016525098


## Add noise on Top 3 important variables

In [8]:
print("sadornot distinct values:", X_train['sadornot'].unique())
print("experiences distinct values:", X_train['experiences'].unique())
print("gpa_all distinct values:", X_train['gpa_all'].unique())

sadornot distinct values: [1. 2.]
experiences distinct values: [5. 4. 3. 2. 1.]
gpa_all distinct values: [3.476 3.705 3.373 3.474 3.947 3.519 3.625 2.4   3.826 3.029 3.79  3.505
 2.815 3.667 3.245 3.293 3.719]


In [9]:
# add noise to variables: sadornot
# Randomly add 1, subtract 1, or make no change for each sample.
noisy_X_train = X_train.copy()

sadornot_noise = np.random.choice([-1, 0, 1], size=noisy_X_train.shape[0])

noisy_X_train['sadornot'] = noisy_X_train['sadornot'] + sadornot_noise
noisy_X_train['sadornot'] = noisy_X_train['sadornot'].clip(lower=1, upper=2)

print(noisy_X_train['sadornot'].value_counts().sort_index())

sadornot
1.0    3847
2.0    3355
Name: count, dtype: int64


In [10]:
# add noise to variable: experiences
# apply normal distribution to choose the reasonable noise to add on the original experience column

experiences_noise = np.random.choice([-1, 0, 1], size=noisy_X_train.shape[0], p=[0.25, 0.5, 0.25])

noisy_X_train['experiences'] = noisy_X_train['experiences'] + experiences_noise
noisy_X_train['experiences'] = noisy_X_train['experiences'].clip(lower=1, upper=5)

print(noisy_X_train['experiences'].head(20))

3185    5.0
1298    4.0
2835    4.0
740     4.0
3784    3.0
7857    2.0
8943    3.0
6814    5.0
7254    2.0
5312    4.0
4078    3.0
8722    4.0
5261    4.0
6590    5.0
8689    5.0
3859    3.0
5112    3.0
435     2.0
8967    2.0
6003    3.0
Name: experiences, dtype: float64


In [11]:
# add noise to variable: gpa_all
# gpa_all is continuous numerical variable
# apply normal distribution to choose the reasonable noise to add on the original sleep_hours column

gpa_noise = np.random.normal(0, 0.05, size=noisy_X_train.shape[0])
noisy_X_train['gpa_all'] = noisy_X_train['gpa_all'] + gpa_noise
print(noisy_X_train['gpa_all'].head(10))

3185    3.511672
1298    3.663540
2835    3.388406
740     3.425625
3784    3.864249
7857    3.576360
8943    3.502484
6814    3.553953
7254    2.433579
5312    3.843116
Name: gpa_all, dtype: float64


## re-train the model with Top3 variables + noise

In [12]:
model_noise_top3 = RandomForestRegressor(**params)
model_noise_top3.fit(noisy_X_train, y_train)

y_pred_noise_top3 = model_noise_top3.predict(X_test)

/opt/miniconda3/lib/python3.12/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


## validation

In [13]:
# MSE
mse_top3 = mean_squared_error(y_test, y_pred_noise_top3)
print("MSE:", mse_top3)

# MAE
mae_top3 = mean_absolute_error(y_test, y_pred_noise_top3)
print("MAE:", mae_top3)

# R-squared
r2_top3 = r2_score(y_test, y_pred_noise_top3)
print("R² Score:", r2_top3)

MSE: 4.090289366562092
MAE: 1.571252048103057
R² Score: 0.8744400316874721


## Add noise on Bottom 3 important variables

In [14]:
print("has_negative_text distinct values:", X_train['has_negative_text'].unique())
print("schedule distinct values:", X_train['schedule'].unique())
print("have distinct values:", X_train['have'].unique())

has_negative_text distinct values: [1. 0.]
schedule distinct values: [1. 2.]
have distinct values: [2. 1.]


In [15]:
less_relevent_noisy_X_train = X_train.copy()

binary_features = {
    'has_negative_text': (0, 1),
    'schedule': (1, 2),
    'have': (1, 2)
}

for feature, (min_val, max_val) in binary_features.items():
    noise = np.random.choice([-1, 0, 1], size=less_relevent_noisy_X_train.shape[0])
    less_relevent_noisy_X_train[feature] = less_relevent_noisy_X_train[feature] + noise
    less_relevent_noisy_X_train[feature] = less_relevent_noisy_X_train[feature].clip(lower=min_val, upper=max_val)

print(less_relevent_noisy_X_train[list(binary_features.keys())].head(10))

      has_negative_text  schedule  have
3185                1.0       2.0   1.0
1298                1.0       1.0   1.0
2835                0.0       2.0   2.0
740                 0.0       2.0   1.0
3784                0.0       1.0   1.0
7857                1.0       1.0   2.0
8943                1.0       1.0   1.0
6814                0.0       2.0   2.0
7254                1.0       1.0   1.0
5312                1.0       1.0   2.0


## re-train the model with bottom 3 variables + noise

In [16]:
model_noise_bottom3 = RandomForestRegressor(**params)
model_noise_bottom3.fit(less_relevent_noisy_X_train, y_train)

y_pred_noise_bottom3 = model_noise_bottom3.predict(X_test)

/opt/miniconda3/lib/python3.12/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


## validation

In [17]:
# MSE
mse_bottom3 = mean_squared_error(y_test, y_pred_noise_bottom3)
print("MSE:", mse_bottom3)

# MAE
mae_bottom3 = mean_absolute_error(y_test, y_pred_noise_bottom3)
print("MAE:", mae_bottom3)

# R-squared
r2_bottom3 = r2_score(y_test, y_pred_noise_bottom3)
print("R² Score:", r2_bottom3)

MSE: 3.6054070385683
MAE: 1.5455121032319195
R² Score: 0.8893245066676314


| Random Forest Model                         | MSE    | % change in MSE | R²     | % change in R² |
|---------------------------------------------|--------|------------------|--------|-----------------|
| No Noise                                    | 3.6133 | /                | 0.8891 | /               |
| Noise on *experiences, gpa_all, sadornot*   | 4.0533 | +12.16%          | 0.8756 | -1.52%          |
| Noise on *have, schedule, has_negative_text*| 3.6056 | -0.21%           | 0.8893 | +0.02%          |
